In [102]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

### 1- Load and Combine

In [44]:
df1=pd.read_csv('train.csv')
df2=pd.read_csv('test.csv')

In [45]:
# concatenate the two dataframes
df = pd.concat([df1, df2], ignore_index=True)
df

,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,...,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,8410,31,Male,19,Education,5390,Excellent,Medium,Average,2,...,0,Mid,Medium,89,No,No,No,Excellent,Medium,Stayed
1,64756,59,Female,4,Media,5534,Poor,High,Low,3,...,3,Mid,Medium,21,No,No,No,Fair,Low,Stayed
2,30257,24,Female,10,Healthcare,8159,Good,High,Low,0,...,3,Mid,Medium,74,No,No,No,Poor,Low,Stayed
3,65791,36,Female,7,Education,3989,Good,High,High,1,...,2,Mid,Small,50,Yes,No,No,Good,Medium,Stayed
4,65026,56,Male,41,Education,4821,Fair,Very High,Average,0,...,0,Senior,Medium,68,No,No,No,Fair,Medium,Stayed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74493,16243,56,Female,42,Healthcare,7830,Poor,Medium,Average,0,...,0,Senior,Medium,60,No,No,No,Poor,Medium,Stayed
74494,47175,30,Female,15,Education,3856,Good,Medium,Average,2,...,0,Entry,Medium,20,No,No,No,Good,Medium,Left
74495,12409,52,Male,5,Education,5654,Good,Very High,Below Average,0,...,4,Mid,Small,7,No,No,No,Good,High,Left
74496,9554,18,Male,4,Education,5276,Fair,High,Average,0,...,3,Mid,Large,5,No,No,No,Poor,High,Stayed


In [65]:
df.to_csv('combined.csv', index=False)

In [46]:
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')

### 2- Preprocessing & Cleaning

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74498 entries, 0 to 74497
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   employee_id               74498 non-null  int64 
 1   age                       74498 non-null  int64 
 2   gender                    74498 non-null  object
 3   years_at_company          74498 non-null  int64 
 4   job_role                  74498 non-null  object
 5   monthly_income            74498 non-null  int64 
 6   work_life_balance         74498 non-null  object
 7   job_satisfaction          74498 non-null  object
 8   performance_rating        74498 non-null  object
 9   number_of_promotions      74498 non-null  int64 
 10  overtime                  74498 non-null  object
 11  distance_from_home        74498 non-null  int64 
 12  education_level           74498 non-null  object
 13  marital_status            74498 non-null  object
 14  number_of_dependents  

In [48]:
# As attrition is binary in the task file so we will convert it from object to binary
df['attrition']=df['attrition'].map({'Stayed': 1, 'Left': 0 })

In [49]:
num_cols=df.select_dtypes(include='number').columns
for col in num_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-'*40)

employee_id
74498
[ 8410 64756 30257 ... 12409  9554 73042]
----------------------------------------
age
42
[31 59 24 36 56 38 47 48 57 30 29 40 19 33 49 51 39 54 23 45 42 53 37 34
 25 41 55 21 28 26 52 50 27 58 43 35 46 44 20 18 22 32]
----------------------------------------
years_at_company
51
[19  4 10  7 41  3 23 16 44  1 12  6 38 22 30  9 37 33 13  5 21 32 34 11
  2 18 27 17 36  8 28 35 14 24 40 15 45 39 20 43 26 29 46 42 25 31 47 48
 49 50 51]
----------------------------------------
monthly_income
9842
[ 5390  5534  8159 ... 11854 11558 12651]
----------------------------------------
number_of_promotions
5
[2 3 0 1 4]
----------------------------------------
distance_from_home
99
[22 21 11 27 71 37 75  5 39 57 51 26 78 30 98 48 17 86 60 10 18 95 23 16
 63 92 58 55 31 32 82  8 74  3 76 47 67 84 14 36 79 52 44 89 64  7 87 24
 15 81 13 35 68 56 73 41 34 29 19 50 62 49 69 33 61 53 72 91 65 93 28 46
 54  9 45 96 94  1 88 25  6 70 12 99  4 97 42 38 90 43 59 85  2 40 20 77
 83 80 66]


In [50]:
for col in num_cols:
    fig1=px.box(df,x=col)
    fig2= px.histogram(df,x=col)
    fig1.show()
    fig2.show()


- Dealing with anomalies values

In [51]:
# calculate percentage of outliers in company tenure
logical_outliers = df[df['company_tenure'] > (df['age'] - 18)]
logical_outliers_pct = (len(logical_outliers) / len(df)) * 100

print(f"Percentage of logical outliers: {logical_outliers_pct:.2f}%")

Percentage of logical outliers: 91.28%


In [52]:
# we will drop the column
df.drop('company_tenure', axis=1, inplace=True)

In [57]:
# only reserve logical values of years_at_company
df=df[df['age'] - df['years_at_company'] >= 18]

In [63]:
df.duplicated().sum()

np.int64(0)

In [58]:
df.describe()

,employee_id,age,years_at_company,monthly_income,number_of_promotions,distance_from_home,number_of_dependents,attrition
count,44686.000000,44686.000000,44686.000000,44686.000000,44686.000000,44686.000000,44686.000000,44686.000000
mean,37346.155306,42.276418,12.596630,7303.541020,0.827798,50.067963,1.648682,0.524773
std,21518.921002,10.728920,9.323265,2156.367314,0.991357,28.518659,1.550317,0.499392
min,2.000000,19.000000,1.000000,1226.000000,0.000000,1.000000,0.000000,0.000000
25%,18792.500000,34.000000,5.000000,5656.000000,0.000000,25.000000,0.000000,0.000000
50%,37342.500000,43.000000,11.000000,7351.000000,0.000000,50.000000,1.000000,1.000000
75%,56038.750000,52.000000,19.000000,8886.000000,1.000000,75.000000,3.000000,1.000000
max,74498.000000,59.000000,41.000000,16149.000000,4.000000,99.000000,6.000000,1.000000


In [59]:
df.describe(include='object')

,gender,job_role,work_life_balance,job_satisfaction,performance_rating,overtime,education_level,marital_status,job_level,company_size,remote_work,leadership_opportunities,innovation_opportunities,company_reputation,employee_recognition
count,44686,44686,44686,44686,44686,44686,44686,44686,44686,44686,44686,44686,44686,44686,44686
unique,2,5,4,4,4,2,5,3,3,3,2,2,2,4,4
top,Male,Technology,Good,High,Average,No,Bachelor’s Degree,Married,Entry,Medium,No,No,No,Good,Low
freq,24549,11571,16908,22272,26808,29987,13433,22325,17909,22292,36130,42507,37445,22220,17758


In [64]:
cat_cols=df.select_dtypes(include='object').columns
for col in cat_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-'*40)

gender
2
['Female' 'Male']
----------------------------------------
job_role
5
['Media' 'Education' 'Technology' 'Finance' 'Healthcare']
----------------------------------------
work_life_balance
4
['Poor' 'Good' 'Fair' 'Excellent']
----------------------------------------
job_satisfaction
4
['High' 'Very High' 'Medium' 'Low']
----------------------------------------
performance_rating
4
['Low' 'High' 'Below Average' 'Average']
----------------------------------------
overtime
2
['No' 'Yes']
----------------------------------------
education_level
5
['Master’s Degree' 'High School' 'Bachelor’s Degree' 'PhD'
 'Associate Degree']
----------------------------------------
marital_status
3
['Divorced' 'Single' 'Married']
----------------------------------------
job_level
3
['Mid' 'Entry' 'Senior']
----------------------------------------
company_size
3
['Medium' 'Small' 'Large']
----------------------------------------
remote_work
2
['No' 'Yes']
----------------------------------------
lead

In [ ]:
# ordering ordinal columns so they sort and compared correctely
ordinal_cols={
    'work_life_balance'  : ['Poor', 'Fair', 'Good', 'Excellent'],
    'job_satisfaction'   : ['Low', 'Medium', 'High', 'Very High'],
    'performance_rating' : ['Low', 'Below Average', 'Average', 'High'],
    'education_level'    : ['High School', 'Associate Degree', "Bachelor's Degree",
                            "Master's Degree", 'PhD'],
    'job_level'          : ['Entry', 'Mid', 'Senior'],
    'company_size'       : ['Small', 'Medium', 'Large'],
    'employee_recognition': ['Low', 'Medium', 'High', 'Very High'],
    'company_reputation' : ['Poor', 'Fair', 'Good', 'Excellent'],
}

for col, order in ordinal_cols.items():
    if col in df.columns:
        df[col]=pd.Categorical(df[col],categories=order,ordered=True)

- Feature engineering 

In [72]:
# age groups for segment analysis
df = df.copy()
df['age_group'] = pd.cut(
    df['age'],
    bins=[18, 25, 35, 45, 60],
    labels=['18–25', '26–35', '36–45', '46–60']
)

In [73]:
# income tier
df['income_tier']=pd.qcut(
    df['monthly_income'],
    q=4,
    labels=['Q1 Low', 'Q2 Mid-Low', 'Q3 Mid-High', 'Q4 High']
)

### 3-EDA

In [80]:
total=len(df)
left=df['attrition'].sum()
stayed=total-left
rate=round(df['attrition'].mean()*100,1)
print(f"Total employees : {total:,}")
print(f"Employees left  : {left:,}  ({rate}%)")
print(f"Employees stayed: {stayed:,}  ({round(100 - rate, 1)}%)")

Total employees : 44,686
Employees left  : 23,450  (52.5%)
Employees stayed: 21,236  (47.5%)


In [ ]:
# attrition by job level
by_level=df.groupby('job_level',observed=True)['attrition'].mean()*100
by_level=by_level.round(1).sort_values(ascending=False)
by_level
#obs-> senior attrition bigger in the whole dataset

job_level
Senior    79.8
Mid       54.4
Entry     36.8
Name: attrition, dtype: float64

In [ ]:
# attrition by remote work
by_remote=df.groupby('remote_work')['attrition'].mean()*100
by_remote=by_remote.round(1)
by_remote
#obs-> remote employees leave the most

remote_work
No     47.1
Yes    75.3
Name: attrition, dtype: float64

In [86]:
# remote vs job level
cross=df.groupby(['job_level','remote_work'],observed=True)['attrition'].mean()*100
cross=cross.round(1).unstack()
cross

remote_work,No,Yes
job_level,,
Entry,30.8,63.0
Mid,48.8,77.3
Senior,76.2,95.2


In [ ]:
# attrition by number of promotions
by_promo=df.groupby('number_of_promotions')['attrition'].mean()*100
by_promo=by_promo.round(1)
by_promo
#obs-> 3+ promotion 77 attrition as employees hit a career ceiling

number_of_promotions
0    50.5
1    51.4
2    51.0
3    75.1
4    77.1
Name: attrition, dtype: float64

In [ ]:
# promo for serionr employees only
senior_pro=df[df['job_level']=='Senior']
senior_pro=senior_pro.groupby('number_of_promotions')['attrition'].mean()*100
senior_pro=senior_pro.round(1)
senior_pro

number_of_promotions
0    78.6
1    79.4
2    78.5
3    93.8
4    96.7
Name: attrition, dtype: float64

In [87]:
df.columns

Index(['employee_id', 'age', 'gender', 'years_at_company', 'job_role',
       'monthly_income', 'work_life_balance', 'job_satisfaction',
       'performance_rating', 'number_of_promotions', 'overtime',
       'distance_from_home', 'education_level', 'marital_status',
       'number_of_dependents', 'job_level', 'company_size', 'remote_work',
       'leadership_opportunities', 'innovation_opportunities',
       'company_reputation', 'employee_recognition', 'attrition', 'age_group',
       'income_tier'],
      dtype='object')

In [98]:
wlb_order = ['Poor', 'Fair', 'Good', 'Excellent']
by_wlb=df.groupby(['work_life_balance',],observed=True)['attrition'].mean()*100
by_wlb=by_wlb.round(1).reindex(wlb_order)
by_wlb
# obs-> excellent wlb has more 

work_life_balance
Poor         39.8
Fair         42.3
Good         59.5
Excellent    64.5
Name: attrition, dtype: float64

In [ ]:
# attrition by work life balance with job level
by_wlb=df.groupby(['work_life_balance','job_level'],observed=True)['attrition'].mean()*100
by_wlb=by_wlb.round(1).unstack()
by_wlb
# obs-> excellent wlb has more attrition in senior level they want growth not balance

job_level,Entry,Mid,Senior
work_life_balance,,,
Excellent,49.3,66.8,89.4
Fair,27.3,43.4,70.9
Good,43.3,62.2,86.0
Poor,23.7,40.7,69.4


In [92]:
# attrition by department
by_role=df.groupby('job_role')['attrition'].mean()*100
by_role=by_role.round(1).sort_values(ascending=False)
by_role

job_role
Technology    53.1
Finance       53.0
Media         52.9
Healthcare    52.4
Education     51.0
Name: attrition, dtype: float64

In [ ]:
# attrition by employee recongition
by_rec=df.groupby('employee_recognition',observed=True)['attrition'].mean()*100
by_rec=by_rec.round(1).sort_values(ascending=False)
by_rec
# obs-> almost no differnce

employee_recognition
Very High    54.0
High         52.4
Low          52.4
Medium       52.4
Name: attrition, dtype: float64

In [100]:
# attrition by age group
by_age=df.groupby('age_group',observed=True)['attrition'].mean()*100
by_age=by_age.round(1)
by_age

age_group
18–25    46.0
26–35    50.4
36–45    53.9
46–60    53.8
Name: attrition, dtype: float64

In [ ]:
# by reputation
by_rep = df.groupby('company_reputation', observed=True)['attrition'].mean() * 100
by_rep = by_rep.round(1).sort_values(ascending=False)
by_rep
#obs-> good reputation compaies loss more people talent is more mobile

company_reputation
Good         57.1
Excellent    55.7
Fair         47.7
Poor         44.3
Name: attrition, dtype: float64

### 4-Visualization


In [111]:
DARK    = "#353b98"
ACCENT  = "#8a91f2"
DANGER  = "#ff6b6b"
SUCCESS = "#43d9a0"
WHITE   = "#ffffff"
TEXT    = "#1a1d3a"

#### Overall Attrition Dount

In [112]:
donut_df = pd.DataFrame({
    "Status": ["Stayed", "Left"],
    "Count":  [stayed, left]
})
 
fig = px.pie(
    donut_df,
    names="Status",
    values="Count",
    hole=0.6,
    color="Status",
    color_discrete_map={"Stayed": ACCENT, "Left": DANGER},
    title="Overall Attrition Rate",
)
fig.update_traces(textfont_color=TEXT)
fig.update_layout(plot_bgcolor=WHITE, paper_bgcolor=WHITE, height=400)
fig.show()

#### Attrition by Job Level

In [113]:
level_df = df.groupby("job_level", observed=True)["attrition"].mean().reset_index()
level_df["rate"] = (level_df["attrition"] * 100).round(1)
level_df["job_level"] = pd.Categorical(level_df["job_level"], categories=["Entry","Mid","Senior"], ordered=True)
level_df = level_df.sort_values("job_level")
 
fig = px.bar(
    level_df,
    x="job_level", y="rate",
    text="rate",
    color="rate",
    color_continuous_scale=[[0, ACCENT], [1, DARK]],
    title="Attrition Rate by Job Level",
    labels={"job_level": "Job Level", "rate": "Attrition Rate (%)"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside", textfont_color=TEXT)
fig.update_coloraxes(showscale=False)
fig.update_yaxes(range=[0, 100], gridcolor="#f0f0f8")
fig.update_layout(plot_bgcolor=WHITE, paper_bgcolor=WHITE, showlegend=False, height=400)
fig.show()

####  Remote work vs Job Level

In [114]:
remote_df = df.groupby(["job_level", "remote_work"], observed=True)["attrition"].mean().reset_index()
remote_df["rate"] = (remote_df["attrition"] * 100).round(1)
remote_df["job_level"] = pd.Categorical(remote_df["job_level"], categories=["Entry","Mid","Senior"], ordered=True)
remote_df = remote_df.sort_values("job_level")
 
fig = px.bar(
    remote_df,
    x="job_level", y="rate",
    color="remote_work", barmode="group",
    text="rate",
    color_discrete_map={"No": DARK, "Yes": DANGER},
    title="Remote Work x Job Level — Highest Risk Combination",
    labels={"job_level": "Job Level", "rate": "Attrition Rate (%)", "remote_work": "Remote Work"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside", textfont_color=TEXT)
fig.update_yaxes(range=[0, 115], gridcolor="#f0f0f8")
fig.update_layout(
    plot_bgcolor=WHITE, paper_bgcolor=WHITE,
    legend=dict(orientation="h", y=-0.2),
    height=420,
)
fig.show()

#### Prompotion heatmap

In [119]:
promo_df = df[df["number_of_promotions"] <= 4].groupby(
    ["job_level", "number_of_promotions"], observed=True
)["attrition"].mean().reset_index()
promo_df["rate"] = (promo_df["attrition"] * 100).round(1)
promo_df["job_level"] = pd.Categorical(promo_df["job_level"], categories=["Entry","Mid","Senior"], ordered=True)
promo_df["number_of_promotions"] = promo_df["number_of_promotions"].astype(str) + " promos"
 
fig = px.density_heatmap(
    promo_df,
    x="number_of_promotions", y="job_level", z="rate",
    color_continuous_scale=[[0, "#eef0ff"], [0.5, ACCENT], [1, DARK]],
    text_auto=True,
    title="Promotions x Job Level — The Career Ceiling",
    labels={"number_of_promotions": "Promotions", "job_level": "Job Level", "rate": "Attrition %"},
)
fig.update_traces(textfont_color=TEXT)
fig.update_layout(plot_bgcolor=WHITE, paper_bgcolor=WHITE, height=380)
fig.show()

In [121]:
wlb_order = ["Poor", "Fair", "Good", "Excellent"]
wlb_df = df.groupby("work_life_balance", observed=True)["attrition"].mean().reset_index()
wlb_df["rate"] = (wlb_df["attrition"] * 100).round(1)
wlb_df["work_life_balance"] = pd.Categorical(wlb_df["work_life_balance"], categories=wlb_order, ordered=True)
wlb_df = wlb_df.sort_values("work_life_balance")
 
fig = px.bar(
    wlb_df,
    x="work_life_balance", y="rate",
    text="rate",
    color="rate",
    color_continuous_scale=[[0, SUCCESS], [0.5, ACCENT], [1, DANGER]],
    title="Work-Life Balance vs Attrition Rate",
    labels={"work_life_balance": "Work-Life Balance", "rate": "Attrition Rate (%)"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside", textfont_color=TEXT)
fig.update_coloraxes(showscale=False)
fig.update_yaxes(range=[0, 80], gridcolor="#f0f0f8")
fig.update_layout(plot_bgcolor=WHITE, paper_bgcolor=WHITE, showlegend=False, height=400)
fig.show()

In [122]:
role_df = df.groupby("job_role")["attrition"].mean().reset_index()
role_df["rate"] = (role_df["attrition"] * 100).round(1)
role_df = role_df.sort_values("rate", ascending=True)
 
fig = px.bar(
    role_df,
    x="rate", y="job_role",
    orientation="h",
    text="rate",
    color="rate",
    color_continuous_scale=[[0, ACCENT], [1, DARK]],
    title="Attrition Rate by Department — Problem is Systemic",
    labels={"job_role": "", "rate": "Attrition Rate (%)"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside", textfont_color=TEXT)
fig.update_coloraxes(showscale=False)
fig.update_xaxes(range=[0, 65], gridcolor="#f0f0f8")
fig.update_layout(plot_bgcolor=WHITE, paper_bgcolor=WHITE, showlegend=False, height=380)
fig.show()

In [123]:
income_df = df[["monthly_income", "attrition"]].copy()
income_df["Status"] = income_df["attrition"].map({0: "Stayed", 1: "Left"})
 
fig = px.histogram(
    income_df,
    x="monthly_income", color="Status",
    barmode="overlay", nbins=40,
    opacity=0.75,
    color_discrete_map={"Stayed": ACCENT, "Left": DANGER},
    title="Monthly Income — Stayed vs Left",
    labels={"monthly_income": "Monthly Income ($)", "count": "Employees"},
)
fig.update_traces(textfont_color=TEXT)
fig.update_yaxes(gridcolor="#f0f0f8")
fig.update_layout(
    plot_bgcolor=WHITE, paper_bgcolor=WHITE,
    legend=dict(orientation="h", y=-0.2),
    height=400,
)
fig.show()

In [ ]:
cost_df = (
    df.groupby("job_level", observed=True)
      .agg(
          attrition_sum=("attrition", "sum"),
          avg_income=("monthly_income", "mean"),
      )
      .reindex(["Entry", "Mid", "Senior"])
      .reset_index()
)
cost_df["cost_M"] = (cost_df["attrition_sum"] * cost_df["avg_income"] * 6 / 1e6).round(1)
cost_df = cost_df[["job_level", "cost_M"]]
 
fig = px.bar(
    cost_df,
    x="job_level", y="cost_M",
    text="cost_M",
    color="job_level",
    color_discrete_map={"Entry": ACCENT, "Mid": DARK, "Senior": DANGER},
    title="Estimated Replacement Cost by Job Level ($M)",
    labels={"job_level": "Job Level", "cost_M": "Estimated Cost ($M)"},
)
fig.update_traces(texttemplate="$%{text}M", textposition="outside", textfont_color=TEXT)
fig.update_yaxes(gridcolor="#f0f0f8")
fig.update_layout(plot_bgcolor=WHITE, paper_bgcolor=WHITE, showlegend=False, height=400)
fig.show()

C:\Users\SOMIA\AppData\Local\Temp\ipykernel_12912\1542443935.py:1: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

